## Connexion

In [0]:
storage_account_name = dbutils.secrets.get(scope="energy-bi-scope", key="blob-storage-account-name")
storage_account_key  = dbutils.secrets.get(scope="energy-bi-scope", key="blob-storage-account-key")
container_name_gold = "gold"

spark.conf.set(
    f"fs.azure.account.key.{storage_account_name}.blob.core.windows.net",
    storage_account_key
)
print("Connexion OK")

## Lire Silver

In [0]:
from pyspark.sql import functions as F

silver_path = f"wasbs://silver@{storage_account_name}.blob.core.windows.net/energy/"

df_silver = (
    spark.read.option("header", "true").option("inferSchema", "true").csv(silver_path)
)

print(f"Silver lignes : {df_silver.count()}")

## Agrégation journalière (Gold)

In [0]:
df_gold = (
    df_silver.groupBy("Date")
    .agg(
        # Énergie totale en Wh (puissance active × 1 minute / 60)
        (F.sum("Global_active_power") * 1000 / 60).alias("energy_wh"),
        F.avg("Global_active_power").alias("Global_active_power_mean"),
        F.avg("Voltage").alias("voltage_mean"),
        F.avg("Global_intensity").alias("intensity_mean"),
        F.sum("Sub_metering_1").alias("sub_metering_1_total"),
        F.sum("Sub_metering_2").alias("sub_metering_2_total"),
        F.sum("Sub_metering_3").alias("sub_metering_3_total"),
        F.count("*").alias("record_count"),
    )
    .withColumnRenamed("Date", "consumption_date")
    .orderBy("consumption_date")
)

print(f"Gold lignes (jours uniques) : {df_gold.count()}")
df_gold.show(5)

## Écrire dans Gold

In [0]:
gold_path = f"wasbs://{container_name_gold}@{storage_account_name}.blob.core.windows.net/energy/"

df_gold.write.mode("overwrite").option("header", "true").csv(gold_path)

print("Données gold écrites avec succès")

## Lire les credentials SQL depuis Key Vault

In [0]:
sql_server = dbutils.secrets.get(scope="energy-bi-scope", key="sql-server-name")
sql_database = dbutils.secrets.get(scope="energy-bi-scope", key="sql-database-name")
sql_username = dbutils.secrets.get(scope="energy-bi-scope", key="sql-username")
sql_password = dbutils.secrets.get(scope="energy-bi-scope", key="sql-password")

jdbc_url = f"jdbc:sqlserver://{sql_server}:1433;database={sql_database};encrypt=true;trustServerCertificate=False;hostNameInCertificate=*.database.windows.net;loginTimeout=30"

print("Credentials SQL chargés")

##  Écrire Gold dans Azure SQL

In [0]:
import time

for attempt in range(3):
    try:
        df_gold.write.format("jdbc").option("url", jdbc_url).option(
            "dbtable", "dbo.FactEnergyGold"
        ).option("user", sql_username).option("password", sql_password).option(
            "driver", "com.microsoft.sqlserver.jdbc.SQLServerDriver"
        ).mode(
            "overwrite"
        ).save()
        print("Table dbo.FactEnergyGold écrite dans Azure SQL")
        break
    except Exception as e:
        print(f"Tentative {attempt+1} échouée. Attente 30s...")
        time.sleep(30)

## Vérifier

In [0]:
df_verify = (
    spark.read.format("jdbc")
    .option("url", jdbc_url)
    .option("dbtable", "dbo.FactEnergyGold")
    .option("user", sql_username)
    .option("password", sql_password)
    .option("driver", "com.microsoft.sqlserver.jdbc.SQLServerDriver")
    .load()
)

print(f"Lignes dans Azure SQL : {df_verify.count()}")
df_verify.show(5)